In [ ]:
import pandas as pd
import numpy as np
from faker import Faker

# Create sample dataset
fake = Faker()
np.random.seed(42)

data = {
    'id': range(1, 101),
    'name': [fake.name() for _ in range(100)],
    'age': np.random.randint(18, 80, 100),
    'salary': np.random.randint(20000, 150000, 100),
    'department': np.random.choice(['Sales', 'IT', 'HR', 'Finance'], 100),
    'join_date': pd.date_range('2020-01-01', periods=100, freq='D'),
    'performance': np.random.choice([np.nan, 'Good', 'Average', 'Excellent'], 100)
}

df = pd.DataFrame(data)

# Data Inspection
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())

## Data Cleaning

### Steps:
1. Handle missing values
2. Remove duplicates
3. Detect and handle outliers
4. Data type conversions
5. Text cleaning

In [ ]:
df_clean = df.copy()

# Handle missing values
print("Handling missing values...")
# Option 1: Drop rows with missing values
# df_clean = df_clean.dropna()

# Option 2: Fill with mean/median
df_clean['performance'].fillna('Not Rated', inplace=True)

print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")

# Remove duplicates
print(f"\nDuplicates before: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()
print(f"Duplicates after: {df_clean.duplicated().sum()}")

# Detect outliers using IQR
Q1 = df_clean['salary'].quantile(0.25)
Q3 = df_clean['salary'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clean[(df_clean['salary'] < Q1 - 1.5*IQR) | (df_clean['salary'] > Q3 + 1.5*IQR)]
print(f"\nOutliers found: {len(outliers)}")

print(f"\nCleaned dataset shape: {df_clean.shape}")

In [ ]:
# Data Transformation
df_transformed = df_clean.copy()

# Feature scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
df_transformed['salary_scaled'] = scaler.fit_transform(df_transformed[['salary']])

# Encoding categorical variables
df_transformed['department_encoded'] = pd.factorize(df_transformed['department'])[0]
df_transformed = pd.get_dummies(df_transformed, columns=['performance'], prefix='perf')

# Feature engineering
df_transformed['experience_years'] = (pd.Timestamp.now() - df_transformed['join_date']).dt.days / 365.25
df_transformed['age_group'] = pd.cut(df_transformed['age'], bins=[0, 30, 50, 100], labels=['Young', 'Middle', 'Senior'])

print("Transformed dataset:")
print(df_transformed.head())

## ETL Pipeline

### Extract → Transform → Load

**Extract**: Get data from sources
**Transform**: Clean and process data
**Load**: Store in data warehouse

### Best Practices:
- Modular design
- Error handling
- Data validation
- Logging and monitoring
- Scalability

In [ ]:
# Simple ETL Pipeline Class
class ETLPipeline:
    def __init__(self):
        self.raw_data = None
        self.transformed_data = None
    
    def extract(self, source):
        """Extract data from source"""
        self.raw_data = pd.read_csv(source) if isinstance(source, str) else source
        print(f"Extracted {len(self.raw_data)} rows")
        return self
    
    def transform(self):
        """Transform data"""
        self.transformed_data = self.raw_data.copy()
        
        # Apply transformations
        self.transformed_data = self.transformed_data.dropna()
        self.transformed_data = self.transformed_data.drop_duplicates()
        
        print(f"Transformed to {len(self.transformed_data)} rows")
        return self
    
    def load(self, destination):
        """Load data to destination"""
        if isinstance(destination, str):
            self.transformed_data.to_csv(destination, index=False)
        print(f"Loaded data to {destination}")
        return self.transformed_data

# Usage
pipeline = ETLPipeline()
result = pipeline.extract(df).transform().load('processed_data.csv')

## Practice Exercises

1. **Load and Explore**: Load a real-world dataset and perform exploratory analysis
2. **Data Cleaning**: Handle missing values, duplicates, and outliers
3. **Feature Engineering**: Create new features from existing ones
4. **Pipeline Building**: Create an ETL pipeline for a specific dataset
5. **Validation**: Implement data validation checks